# Семинар 12. NumPy и Pandas

В данных — продажи эспрессо, американо, капучино и латте за 28 дней.
Посчитайте продажи и выручку, найдите лучшие дни и составьте таблицу итогов.

Выполняйте задания по порядку: следующие используют результаты предыдущих.
Заменяйте `...` своим кодом и запускайте проверку под заданием.
Если изменили исходные данные, перезапустите ядро и выполните ячейки заново.

В заданиях 1–10 и 15 используйте NumPy без циклов по элементам,
в заданиях 11–14 — Pandas. Задания 16–17 — дополнительные.


## Загрузка данных

Установите зависимости из корня репозитория:

```bash
uv sync --extra notebook
```

Выберите ядро Python из `.venv`. Выполните следующую ячейку загрузки.
CSV уже лежат в `seminar12/data/`.

Переменные:

- `cups` — общее число чашек за каждый день;
- `sales` — продажи: строки соответствуют дням, столбцы — напиткам;
- `prices` — цены напитков;
- `dates` — даты;
- `drinks` — названия напитков в порядке столбцов `sales`;
- `weekend_mask` — маска суббот и воскресений.

Другой набор данных можно создать командой из [README](README.md).


In [ ]:
import csv
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

# Поддерживаются запуск из корня репозитория и запуск из seminar12.
seminar_dir = Path.cwd() if Path("generate_data.py").is_file() else Path.cwd() / "seminar12"
data_dir = seminar_dir / "data"

with (data_dir / "prices.csv").open(encoding="utf-8", newline="") as stream:
    price_rows = list(csv.DictReader(stream))
with (data_dir / "sales.csv").open(encoding="utf-8", newline="") as stream:
    sale_rows = list(csv.DictReader(stream))

drinks = [row["drink"] for row in price_rows]
prices = np.array([float(row["price"]) for row in price_rows])
dates = np.array([row["date"] for row in sale_rows])
raw_sales = [[int(row[drink]) for drink in drinks] for row in sale_rows]
sales = np.array(raw_sales, dtype=np.int64)
cups = np.array([sum(row) for row in raw_sales], dtype=np.int64)
weekend_mask = np.array([date.fromisoformat(day).weekday() >= 5 for day in dates])

print("Напитки:", drinks)
print("Цены:", prices)
print("Даты:", dates[:7])
print("Всего чашек за день:", cups[:7])

## 1. Одномерный массив

`ndarray` — массив элементов одного типа. `shape` — кортеж размеров по осям,
`ndim` — число осей, `dtype` — тип элементов. У одномерного массива форма `(n,)`.

Арифметические операции выполняются поэлементно:
`items * 2` повторяет список, но удваивает числа в массиве NumPy.
Предскажите вывод примера перед запуском.


In [ ]:
example = np.array([10, 20, 30])
print([10, 20, 30] * 2)
print(example * 2)
print(cups.shape, cups.ndim, cups.dtype)

### Задание 1. Удвоение продаж

Получите `doubled` — массив продаж каждого дня, умноженных на два.
Не меняйте `cups`. Какой формы будет результат?


In [ ]:
doubled = ...

In [ ]:
assert isinstance(doubled, np.ndarray), "Нужен массив NumPy"
assert doubled.shape == cups.shape
assert doubled.tolist() == [value * 2 for value in cups.tolist()]
assert cups.tolist() == [sum(row) for row in raw_sales], "Исходные продажи изменились"


## 2. Индексы, срезы и копирование

Индексы начинаются с нуля. `a[-1]` — последний элемент.
Срез `a[start:stop:step]` не включает `stop`; `a[:]` выбирает весь массив.

Например, `a[1:4]` берёт элементы 1, 2, 3, `a[::2]` — каждый второй,
`a[::-1]` — все элементы в обратном порядке.


### Задание 2. Срезы

Получите срезы `cups`:

- `first_three` — первые три дня;
- `last_two` — последние два дня;
- `every_other` — каждый второй день, начиная с первого.


In [ ]:
first_three = ...
last_two = ...
every_other = ...

In [ ]:
assert isinstance(first_three, np.ndarray)
assert isinstance(last_two, np.ndarray)
assert isinstance(every_other, np.ndarray)
assert first_three.tolist() == cups.tolist()[:3]
assert last_two.tolist() == cups.tolist()[-2:]
assert every_other.tolist() == cups.tolist()[::2]


Срез NumPy — представление (`view`), использующее память исходного массива.
Изменение среза меняет исходные данные. `.copy()` создаёт независимую копию.
Одно число, полученное по индексу из одномерного числового массива,
не меняется при последующей записи в массив.

Предскажите вывод следующей ячейки.


In [ ]:
sample = np.array([10, 20, 30, 40])
window = sample[1:3]
separate = sample[1:3].copy()
number = sample[1]
sample[1] = 99
print("Исходный:", sample)
print("Срез:", window)
print("Копия:", separate)
print("Одно число:", number)

### Задание 3. Независимая копия

Создайте `safe_fragment` — копию первых трёх элементов `cups`.
Следующая строка обнуляет первый элемент копии. `cups` должен остаться прежним.


In [ ]:
safe_fragment = ...
safe_fragment[0] = 0

In [ ]:
assert isinstance(safe_fragment, np.ndarray)
assert not np.shares_memory(safe_fragment, cups), "Получился срез с общей памятью"
assert safe_fragment.tolist() == [0, *cups.tolist()[1:3]]
assert cups.tolist() == [sum(row) for row in raw_sales], "cups изменился: загрузите данные заново"


## 3. Булевы маски

Сравнение массива с числом даёт массив `bool` той же формы — маску.
`a[mask]` выбирает элементы с отметкой `True`.
`mask.sum()` считает число `True`; `int(...)` преобразует результат в обычный `int`.

Условия соединяются через `&` (и), `|` (или), `~` (не).
Каждое сравнение берите в скобки: `(a >= 10) & (a < 20)`.


In [ ]:
example = np.array([5, 12, 8, 20])
mask = example < 10
print(mask)
print(example[mask])
print(int(mask.sum()))

### Задание 4. Маска тихих дней

Тихий день — день с продажами строго меньше 170 чашек.
Получите булеву маску `quiet_mask` той же формы, что и `cups`.


In [ ]:
quiet_limit = 170
quiet_mask = ...

In [ ]:
assert isinstance(quiet_mask, np.ndarray)
assert quiet_mask.dtype == np.bool_
assert quiet_mask.shape == cups.shape
assert quiet_mask.tolist() == [value < quiet_limit for value in cups.tolist()]


### Задание 5. Отбор и подсчёт

Получите `quiet_days` — продажи тихих дней, и `quiet_count` — число таких дней
как обычный `int`. Что получится, если тихих дней нет?


In [ ]:
quiet_days = ...
quiet_count = ...

In [ ]:
assert isinstance(quiet_days, np.ndarray)
assert type(quiet_count) is int
assert quiet_days.tolist() == [value for value in cups.tolist() if value < quiet_limit]
assert quiet_count == len(quiet_days)
print("Тихих дней:", quiet_count, "; продажи:", quiet_days)

## 4. Двумерный массив и оси

В `sales` строка — день, столбец — напиток в порядке `drinks`.
Для исходных данных форма равна `(28, 4)`.

`a[i, j]` — элемент, `a[i, :]` — строка, `a[:, j]` — столбец.
Двоеточие выбирает все элементы оси. Булева маска строк выбирает строки целиком.


In [ ]:
print("Форма:", sales.shape)
print("Первые дни:", sales[:3])
print("Порядок столбцов:", drinks)

### Задание 6. Столбец и строки

Получите из `sales`:

- `espresso_sales` — продажи первого напитка за все дни;
- `weekend_sales` — строки суббот и воскресений по маске `weekend_mask`.

Какой формы будет каждый результат?


In [ ]:
espresso_sales = ...
weekend_sales = ...

In [ ]:
assert isinstance(espresso_sales, np.ndarray)
assert isinstance(weekend_sales, np.ndarray)
assert espresso_sales.shape == (len(dates),)
assert espresso_sales.tolist() == [row[0] for row in raw_sales]
assert weekend_sales.shape == (int(weekend_mask.sum()), len(drinks))
assert weekend_sales.tolist() == [
    row for row, flag in zip(raw_sales, weekend_mask, strict=False) if flag
]
print("Формы:", espresso_sales.shape, weekend_sales.shape)

### Редукции по осям

`sum()` складывает все элементы. `sum(axis=...)` убирает указанную ось:

- `axis=0` — складывает дни, оставляет итог по каждому напитку;
- `axis=1` — складывает напитки, оставляет итог по каждому дню.

Так же работают `mean`, `min`, `max`. Перед вычислением определите форму ответа.


In [ ]:
small = np.array([[1, 2, 3], [10, 20, 30]])
print(small.sum())
print(small.sum(axis=0))
print(small.sum(axis=1))

### Задание 7. Суммы по дням и напиткам

Получите `daily_totals` — число чашек каждого дня,
и `drink_totals` — продажи каждого напитка за весь период.
Суммы обоих результатов должны совпадать.


In [ ]:
daily_totals = ...
drink_totals = ...

In [ ]:
assert isinstance(daily_totals, np.ndarray)
assert isinstance(drink_totals, np.ndarray)
assert daily_totals.shape == (len(dates),)
assert drink_totals.shape == (len(drinks),)
assert daily_totals.tolist() == cups.tolist()
assert drink_totals.tolist() == [sum(column) for column in zip(*raw_sales, strict=False)]
assert daily_totals.sum() == drink_totals.sum()
print("Всего по напиткам:", dict(zip(drinks, drink_totals.tolist(), strict=False)))

`max()` возвращает значение максимума, `argmax()` — индекс первого максимума.
По этому индексу можно получить дату из `dates`.
Для минимума есть `min()` и `argmin()`.


### Задание 8. Средние продажи и максимум

Получите `drink_means` — средние дневные продажи каждого напитка.
Найдите `best_day` — индекс дня с наибольшим числом чашек, обычный `int`.
При равенстве выберите первый день. Выведите его дату.


In [ ]:
drink_means = ...
best_day = ...

In [ ]:
assert isinstance(drink_means, np.ndarray)
assert drink_means.shape == (len(drinks),)
np.testing.assert_allclose(
    drink_means, [sum(col) / len(raw_sales) for col in zip(*raw_sales, strict=False)]
)
assert type(best_day) is int
assert best_day == cups.tolist().index(max(cups.tolist()))
print("Лучший день:", dates[best_day], "; чашек:", daily_totals[best_day])

## 5. Broadcasting

У `sales` форма `(число дней, 4)`, у `prices` — `(4,)`.
При умножении вектор цен применяется к каждой строке продаж.

Это broadcasting. Формы сравниваются справа налево: размеры должны совпадать
или один из них должен быть равен 1. Отсутствующие слева оси считаются единичными.

Например, `(3, 4)` совместима с `(4,)`, но несовместима с `(3,)`.


In [ ]:
small_sales = np.array([[2, 3], [4, 1], [0, 5]])
small_prices = np.array([100.0, 200.0])
print(small_sales.shape, small_prices.shape)
print(small_sales * small_prices)

### Задание 9. Выручка по напиткам

Получите `revenue` той же формы, что и `sales`: количество чашек,
умноженное на цену соответствующего напитка.
Используйте одно умножение без ручного повторения цен.


In [ ]:
revenue = ...

In [ ]:
assert isinstance(revenue, np.ndarray)
assert revenue.shape == sales.shape
expected = [
    [quantity * price for quantity, price in zip(row, prices.tolist(), strict=False)]
    for row in raw_sales
]
np.testing.assert_allclose(revenue, expected)
print("Выручка за первые дни:", revenue[:3])

### Задание 10. Дневная выручка

Получите `daily_revenue` — выручку каждого дня по всем напиткам.
Какую ось нужно убрать?


In [ ]:
daily_revenue = ...

In [ ]:
assert isinstance(daily_revenue, np.ndarray)
assert daily_revenue.shape == cups.shape
np.testing.assert_allclose(daily_revenue, [sum(row) for row in revenue.tolist()])
print("Выручка за период:", float(daily_revenue.sum()), "руб.")

## 6. Pandas

`DataFrame` — таблица с именами колонок и индексом строк, `Series` — одна колонка.
В словаре для `DataFrame` ключи задают названия колонок,
а значения — их содержимое. Длины колонок должны совпадать.

```python
example = pd.DataFrame({"name": ["Аня", "Боря"], "score": [7, 9]})
example["score"]    # Series
example[["score"]]  # DataFrame из одной колонки
example.head(1)     # первая строка
```

`pd.read_csv(path)` читает CSV в `DataFrame`.


### Задание 11. Таблица итогов

Создайте `report` с колонками в указанном порядке:

- `date` — даты из `dates`;
- `cups` — число чашек из `daily_totals`;
- `revenue` — выручка из `daily_revenue`.

Выведите первые пять строк через `.head()`.


In [ ]:
report = ...

In [ ]:
assert isinstance(report, pd.DataFrame)
assert list(report.columns) == ["date", "cups", "revenue"]
assert report.shape == (len(dates), 3)
assert report["date"].tolist() == dates.tolist()
assert report["cups"].tolist() == cups.tolist()
np.testing.assert_allclose(report["revenue"], daily_revenue)
report.head()

`table[table["score"] >= 8]` выбирает строки по условию.
Для нескольких условий используйте `&`, `|` и скобки.
Метки индекса сохраняются.


### Задание 12. Фильтрация строк

Получите `busy` — строки `report` с продажами не меньше 200 чашек.
Сохраните все колонки и исходный порядок строк. Результат может быть пустым.


In [ ]:
busy = ...

In [ ]:
assert isinstance(busy, pd.DataFrame)
assert list(busy.columns) == list(report.columns)
expected_positions = [i for i, value in enumerate(cups.tolist()) if value >= 200]
assert busy.index.tolist() == expected_positions
pd.testing.assert_frame_equal(busy, report.iloc[expected_positions])
busy.head()

`sort_values("column", ascending=False)` сортирует по убыванию.
`kind="stable"` сохраняет исходный порядок при равных значениях.
`.head(k)` берёт первые `k` строк. Метки индекса после сортировки сохраняются.


### Задание 13. Три дня с наибольшей выручкой

Получите `top_days` — первые три строки `report` после сортировки по убыванию выручки.
При равенстве сохраните исходный порядок. Если строк меньше трёх, оставьте все.


In [ ]:
top_days = ...

In [ ]:
assert isinstance(top_days, pd.DataFrame)
expected_positions = sorted(range(len(dates)), key=lambda i: -float(daily_revenue[i]))[:3]
pd.testing.assert_frame_equal(top_days, report.iloc[expected_positions])
top_days

`.iloc[0]` берёт первую строку по позиции, `.loc[0]` — строку с меткой 0.
После сортировки это могут быть разные строки. Предскажите вывод примера.


In [ ]:
example_report = pd.DataFrame({"day": ["пн", "вт", "ср"], "revenue": [100, 300, 200]})
ordered = example_report.sort_values("revenue", ascending=False)
print(ordered)
print("Первая по позиции:", ordered.iloc[0]["day"])
print("С меткой 0:", ordered.loc[0, "day"])

`table["new"] = ...` добавляет или заменяет колонку.
Операции над числовыми колонками выполняются поэлементно.
Чтобы сохранить исходную таблицу, изменяйте её `.copy()`.


### Задание 14. Новая колонка

Создайте копию `report` под именем `report_in_thousands`.
Добавьте в неё `revenue_thousands` — выручку в тысячах рублей.
Не меняйте `report`.


In [ ]:
report_in_thousands = ...
# Добавьте колонку revenue_thousands.

In [ ]:
assert isinstance(report_in_thousands, pd.DataFrame)
assert report_in_thousands is not report
assert list(report.columns) == ["date", "cups", "revenue"]
assert list(report_in_thousands.columns) == [*report.columns, "revenue_thousands"]
pd.testing.assert_frame_equal(report_in_thousands[list(report.columns)], report)
np.testing.assert_allclose(report_in_thousands["revenue_thousands"], daily_revenue / 1000)
report_in_thousands.head()

## 7. Новые цены

Эспрессо остаётся по прежней цене, американо дорожает на 20 рублей,
капучино — на 30, латте — на 40. Число проданных чашек считаем прежним.


### Задание 15. Прирост выручки

По заданному `new_prices` посчитайте:

- `new_daily_revenue` — дневную выручку с новыми ценами;
- `gain` — прирост относительно `daily_revenue`;
- `best_gain_day` — индекс дня с наибольшим приростом, обычный `int`.
  При равенстве выберите первый день.

Не меняйте `sales` и `prices`.
Выведите дату выбранного дня, прежнюю и новую выручку.


In [ ]:
new_prices = prices + np.array([0, 20, 30, 40])
new_daily_revenue = ...
gain = ...
best_gain_day = ...

In [ ]:
assert isinstance(new_daily_revenue, np.ndarray)
assert isinstance(gain, np.ndarray)
assert new_daily_revenue.shape == gain.shape == cups.shape
expected_new = [
    sum(q * p for q, p in zip(row, new_prices.tolist(), strict=False)) for row in raw_sales
]
np.testing.assert_allclose(new_daily_revenue, expected_new)
np.testing.assert_allclose(
    gain, [new - old for new, old in zip(expected_new, daily_revenue.tolist(), strict=False)]
)
assert type(best_gain_day) is int
assert best_gain_day == gain.tolist().index(max(gain.tolist()))
assert sales.tolist() == raw_sales
assert prices.tolist() == [float(row["price"]) for row in price_rows]
print("Дата:", dates[best_gain_day], "; прирост:", float(gain[best_gain_day]), "руб.")

## Дополнительные задания

`groupby` объединяет строки с одинаковым значением колонки в группы.
Затем к каждой группе применяется операция.
Например, `table.groupby("group")["score"].mean()` считает средний балл по группам.


### Задание 16. Средняя выручка по типу дня

Создайте копию `report` под именем `by_day_type`.
Добавьте `is_weekend` из `weekend_mask`.
Получите `mean_by_type` — `Series` средней дневной выручки для будней (`False`)
и выходных (`True`). Если в данных нет дней одного из типов, группы для него не будет.


In [ ]:
by_day_type = ...
mean_by_type = ...

In [ ]:
assert isinstance(by_day_type, pd.DataFrame)
assert by_day_type["is_weekend"].tolist() == weekend_mask.tolist()
assert isinstance(mean_by_type, pd.Series)
assert mean_by_type.index.tolist() == sorted(set(weekend_mask.tolist()))
for flag, mean_value in mean_by_type.items():
    group = [
        value
        for value, is_weekend in zip(daily_revenue.tolist(), weekend_mask.tolist(), strict=False)
        if is_weekend == flag
    ]
    assert np.isclose(mean_value, sum(group) / len(group))
assert "is_weekend" not in report.columns
mean_by_type

`np.where(condition, yes, no)` выбирает значение для каждого элемента:
`np.where(np.array([True, False]), 0.9, 1.0)` даёт `[0.9, 1.0]`.

У дневных коэффициентов форма `(n,)`, у выручки — `(n, 4)`.
Чтобы умножить каждую строку на её коэффициент, нужна форма `(n, 1)`.
`a[:, np.newaxis]` добавляет ось длины 1; `None` означает то же самое.


### Задание 17. Скидка в тихие дни

В тихие дни все напитки дешевле на 10%. Число проданных чашек не меняется.
Через `np.where` создайте `factors` — дневные коэффициенты 0.9 или 1.0.
Получите `discounted_revenue` формы `sales.shape`, умножив строки `revenue`
на соответствующие коэффициенты.


In [ ]:
factors = ...
discounted_revenue = ...

In [ ]:
assert isinstance(factors, np.ndarray)
assert factors.shape == cups.shape
assert isinstance(discounted_revenue, np.ndarray)
assert discounted_revenue.shape == sales.shape
expected_factors = [0.9 if value < quiet_limit else 1.0 for value in cups.tolist()]
np.testing.assert_allclose(factors, expected_factors)
np.testing.assert_allclose(
    discounted_revenue,
    [
        [value * factor for value in row]
        for row, factor in zip(revenue.tolist(), expected_factors, strict=False)
    ],
)
print("Выручка после скидок:", float(discounted_revenue.sum()), "руб.")

## Проверьте себя

- Как меняется форма `sales` при суммировании по каждой оси?
- Чем маска отличается от выбранных по ней значений?
- Когда нужна `.copy()`?
- Почему для умножения на дневные коэффициенты нужна новая ось?
- Чем `max()` отличается от `argmax()`, а `loc` — от `iloc`?

Прочитайте исходный CSV в Pandas. Сравните его колонки с колонками `report`.


In [ ]:
source_table = pd.read_csv(data_dir / "sales.csv")
source_table.head()